# SilentSpeech: render every phrase in a cloned voice (XTTS-v2)

Run this on Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**, then run the cells in order.

**Before you start**
1. **Consent.** The voice owner has signed `docs/consent_form.md` and agreed to how their voice is used.
2. **License.** XTTS-v2 is released under the Coqui Public Model License, which allows non-commercial use only. Running the model cell confirms you accept it.
3. **Files you need:** a clean voice sample (30-60 s of natural speech in a quiet room, WAV/MP3/M4A) and `phrases/phrases_en.json` from the repo. Several samples of the same person are fine.

**Output:** a zip with one WAV per phrase. Unzip it into `audio/phrase_audio/` so files end up at `audio/phrase_audio/<speaker>/P01.wav`.

In [ ]:
# 1. Install the maintained Coqui TTS package (idiap/coqui-ai-TTS).
#    If Colab asks you to restart the session afterwards, do it and continue with the next cell.
!pip install -q coqui-tts
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to a GPU")

In [ ]:
# 2. Settings
SPEAKER_ID = "s01"   # must match --speaker in app.py
LANGUAGE = "en"      # XTTS-v2: en hi es fr de it pt pl tr ru nl cs ar zh-cn ja ko hu
PHRASE_SET = "full"  # "mvp" (P01-P30) or "full" (all 50)

In [ ]:
# 3. Upload the voice sample(s) and the phrase file
import json
from google.colab import files

print("Select the voice sample(s) and phrases_en.json")
uploaded = files.upload()
phrase_file = next(n for n in uploaded if n.endswith(".json"))
voice_files = [n for n in uploaded if n.lower().endswith((".wav", ".mp3", ".m4a", ".flac", ".ogg"))]
assert voice_files, "Upload at least one voice sample"

with open(phrase_file, encoding="utf-8") as f:
    phrases = json.load(f)
if PHRASE_SET == "mvp":
    phrases = [p for p in phrases if p.get("set", "mvp") == "mvp"]
print(f"{len(phrases)} phrases, voice samples: {voice_files}")

In [ ]:
# 4. Load XTTS-v2 (downloads about 1.8 GB the first time)
import os
os.environ["COQUI_TOS_AGREED"] = "1"  # you accept the Coqui Public Model License (non-commercial)

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

In [ ]:
# 5. Render one WAV per phrase
out_dir = f"phrase_audio/{SPEAKER_ID}"
os.makedirs(out_dir, exist_ok=True)

def spoken(text):
    # A full stop helps XTTS end very short phrases ("Yes", "No") cleanly.
    text = text.strip()
    return text if text[-1] in ".!?" else text + "."

def render(p):
    tts.tts_to_file(text=spoken(p["text"]), speaker_wav=voice_files, language=LANGUAGE,
                    file_path=f"{out_dir}/{p['id']}.wav")

for p in phrases:
    render(p)
    print("done", p["id"], p["text"])

In [ ]:
# 6. Listen. Short phrases sometimes come out odd; list them in REDO and run again for a new take.
from IPython.display import Audio, display

LISTEN = [p["id"] for p in phrases]   # or e.g. ["P16", "P17", "P21"]
for p in phrases:
    if p["id"] in LISTEN:
        print(p["id"], p["text"])
        display(Audio(f"{out_dir}/{p['id']}.wav"))

In [ ]:
REDO = []   # e.g. ["P16", "P17"]
for p in phrases:
    if p["id"] in REDO:
        render(p)
        print(p["id"], p["text"])
        display(Audio(f"{out_dir}/{p['id']}.wav"))

In [ ]:
# 7. Download a zip. Unzip it into audio/phrase_audio/ in the project.
import shutil
zip_path = shutil.make_archive(f"phrase_audio_{SPEAKER_ID}", "zip", "phrase_audio")
files.download(zip_path)

**Next:** unzip into `audio/phrase_audio/` (you should see `audio/phrase_audio/s01/P01.wav`), then run

```bash
python app.py --speaker s01 --voice cloned
```

The app labels this output as an AI-generated voice. It can only play these fixed phrases, never arbitrary text.
Optionally, watermark the files with Meta's AudioSeal so they can be identified as synthetic.